In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam

2026-02-14 23:36:28.765477: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-14 23:36:28.799876: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-14 23:36:29.680078: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
def build_inception_bilstm_model(input_shape, num_classes, learning_rate=1e-3):
    """
    Arquitectura Híbrida Inception-BiLSTM para EMG (MS-CLSTM style).
    Optimizada para GPUs NVIDIA A100 (sin recurrent_dropout).
    """
    
    # 1. ENTRADA
    inputs = Input(shape=input_shape, name="emg_input")
    
    # --- BLOQUE 1: EXTRACTOR DE CARACTERÍSTICAS (Inception 1D) ---
    # Captura patrones a múltiples escalas (corto, medio, largo plazo) simultáneamente.
    
    # Rama A: Visión "Lupa" (Kernel 1) -> Relaciones instantáneas entre sensores
    branch_a = layers.Conv1D(filters=32, kernel_size=1, padding='same', activation='relu')(inputs)
    
    # Rama B: Visión "Detalle" (Kernel 3) -> Patrones rápidos/transitorios
    branch_b = layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(inputs)
    
    # Rama C: Visión "Panorámica" (Kernel 5) -> La forma general del gesto
    branch_c = layers.Conv1D(filters=32, kernel_size=5, padding='same', activation='relu')(inputs)
    
    # Fusión de Escalas
    x = layers.Concatenate(axis=-1, name="inception_concat")([branch_a, branch_b, branch_c])
    x = layers.BatchNormalization()(x) # Estabiliza la fusión
    x = layers.Activation('relu')(x)
    
    # --- BLOQUE 2: REDUCCIÓN (Pooling) ---
    # OJO: Usamos MaxPooling1D local (no global) para mantener la secuencia temporal viva.
    # Reduce el ruido y la carga computacional a la mitad, pero deja el tiempo intacto.
    x = layers.MaxPooling1D(pool_size=2, padding='same', name="temporal_pool")(x)
    x = layers.Dropout(0.3)(x) # Regularización leve
    
    # --- BLOQUE 3: MEMORIA TEMPORAL (Bidirectional LSTM) ---
    # Bi-LSTM mira la secuencia de izquierda a derecha y viceversa.
    # NOTA IMPORTANTE: No usamos 'recurrent_dropout' para aprovechar CuDNN en A100.
    # return_sequences=False -> Solo nos importa la conclusión final tras ver toda la ventana.
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=False), name="bi_lstm")(x)
    
    # --- BLOQUE 4: CABEZA DE CLASIFICACIÓN (Classifier Head) ---
    x = layers.Dropout(0.5)(x) # Dropout fuerte aquí para evitar memorización
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # Salida Final (Probabilidades)
    outputs = layers.Dense(num_classes, activation='softmax', name="prediction")(x)
    
    # Construir Modelo
    model = Model(inputs=inputs, outputs=outputs, name="MyoTensor_Inception_BiLSTM")
    
    # Compilación
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, 
                  loss='categorical_crossentropy', 
                  metrics=['accuracy'])
    
    return model

# --- INSTANCIACIÓN ---
# Tu ventana de 40 muestras @ 200Hz = 200ms (Perfecto según literatura)
input_shape = (40, 8) 
num_classes = 7

model = build_inception_bilstm_model(input_shape, num_classes)
model.summary()

I0000 00:00:1771122992.211171   18530 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2279 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "MyoTensor_Inception_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ emg_input           │ (None, 40, 8)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 40, 32)    │        288 │ emg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 40, 32)    │        800 │ emg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 40, 32)    │      1,312 │ emg_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inception_concat    │ (None, 40, 96)    │          0 │ conv1d[0][0],     │
│ (Concatenate)       │                   │            │ conv1d_1[0][0],   │
│                     │                   │            │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 40, 96)    │        384 │ inception_concat… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 40, 96)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_pool       │ (None, 20, 96)    │          0 │ activation[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 20, 96)    │          0 │ temporal_pool[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bi_lstm             │ (None, 256)       │    230,400 │ dropout[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ bi_lstm[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │     16,448 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ prediction (Dense)  │ (None, 7)         │        455 │ batch_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 250,343 (977.90 KB)

 Trainable params: 250,023 (976.65 KB)

 Non-trainable params: 320 (1.25 KB)